# **Teachable Machine**

En esta práctica se debe exportar el modelo que se genera en Teachable Machine.

A la hora de exportar el modelo, se puede exportar para Tensorflow.

También viene un fragmento de código para usar el modelo:

```
from keras.models import load_model  # TensorFlow is required for Keras to work
from PIL import Image, ImageOps  # Install pillow instead of PIL
import numpy as np

# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

# Load the model
model = load_model("keras_Model.h5", compile=False)

# Load the labels
class_names = open("labels.txt", "r").readlines()

# Create the array of the right shape to feed into the keras model
# The 'length' or number of images you can put into the array is
# determined by the first position in the shape tuple, in this case 1
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

# Replace this with the path to your image
image = Image.open("<IMAGE_PATH>").convert("RGB")

# resizing the image to be at least 224x224 and then cropping from the center
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

# turn the image into a numpy array
image_array = np.asarray(image)

# Normalize the image
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

# Load the image into the array
data[0] = normalized_image_array

# Predicts the model
prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

# Print prediction and confidence score
print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)

```

**Tarea**: subir 3 imágenes propias (una clara, una ambigua, una que no pertenezca a ninguna categoría) y anotar qué predice y con qué confianza.

**¡Ojo! Hay que cargar una  versión anterior de Keras**

In [1]:
# forzar TensorFlow a usar el comportamiento clásico de Keras 2
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import tf_keras as keras

print("Versión de TensorFlow:", tf.__version__)
print("Versión de TF-Keras (Legacy):", keras.__version__)

try:
    # Cargamos el modelo utilizando el módulo de compatibilidad oficial tf_keras
    model = keras.models.load_model('keras_model.h5', compile=False)
    print("\n¡MODELO CARGADO CON ÉXITO!")
    model.summary()
except Exception as e:
    print(f"\nError al cargar: {e}")

Versión de TensorFlow: 2.21.0
Versión de TF-Keras (Legacy): 2.21.0

¡MODELO CARGADO CON ÉXITO!
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 2)                 128300    
                                                                 
Total params: 538508 (2.05 MB)
Trainable params: 524428 (2.00 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________


In [2]:
import tensorflow as tf
from keras.models import load_model
from PIL import Image, ImageOps
import numpy as np

class_names = open("labels.txt", "r").readlines()
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
image = Image.open("incidente.png").convert("RGB")
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
image_array = np.asarray(image)
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
data[0] = normalized_image_array

prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

# Print prediction and confidence score
print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)

FileNotFoundError: [Errno 2] No such file or directory: 'incidente.png'

In [3]:
import numpy as np
from PIL import Image, ImageOps

class_names = open("labels.txt", "r").readlines()

def clasificar(ruta_imagen):
    data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
    image = Image.open(ruta_imagen).convert("RGB")
    image = ImageOps.fit(image, (224, 224), Image.Resampling.LANCZOS)
    data[0] = (np.asarray(image).astype(np.float32) / 127.5) - 1

    prediction = model.predict(data)
    index = np.argmax(prediction)
    clase = class_names[index][2:].strip()
    confianza = prediction[0][index]

    print(f"Imagen: {ruta_imagen}")
    print(f"Clase predicha: {clase}")
    print(f"Confianza: {confianza:.2%}")
    for i, nombre in enumerate(class_names):
        print(f"  {nombre[2:].strip()}: {prediction[0][i]:.2%}")

In [4]:
# implementa la clasificación con la imagen que pertenece claramente a una categoría
clasificar("clara.jpg")

1/1 [==============================] - 1s 1s/step
Imagen: clara.jpg
Clase predicha: Teclado
Confianza: 100.00%
  Teclado: 100.00%
  Ratón: 0.00%


In [5]:
# implementa la clasificación con una imagen ambigua, que puede ser de cualquier categoría
clasificar("ambigua.jpg")

1/1 [==============================] - 0s 69ms/step
Imagen: ambigua.jpg
Clase predicha: Teclado
Confianza: 99.79%
  Teclado: 99.79%
  Ratón: 0.21%


In [6]:
# implementa la clasificación con una imagen sin ninguna categoría del modelo
clasificar("ninguna.jpg")

1/1 [==============================] - 0s 41ms/step
Imagen: ninguna.jpg
Clase predicha: Ratón
Confianza: 73.05%
  Teclado: 26.95%
  Ratón: 73.05%


**¿Cómo se usa el modelo exportado? ¿Cómo se comporta?**

---



Se exporta el modelo y se carga en Python, y al darle una imagen te dice qué clase es y con qué probabilidad. Con la foto clara acertó: Teclado con 100 %. Con la ambigua dijo Teclado con 99,79 % y con la que no era de ninguna clase dijo Ratón con 73,05 %, porque solo sabe elegir entre Teclado y Ratón.
